### Kullback-Leibler Divergence

This section deals with calculating the Kullback-Leibler divergence to make a comparison between the different trained models.

In [ ]:
# Evaluation of Neural Network PDF Architectures via KL Divergence

import os
import pandas as pd
import numpy as np
from tqdm import tqdm

# --- PDF Parton Distribution Functions: KL Divergence Analysis ---

data_path = "data/"

# Load training data and targets
x_q2_inputs = np.load(os.path.join(data_path, "x_q2_inputs_training.npy"))
val_pdf = np.load(os.path.join(data_path, "pdf_targets_training.npy"))
pids = list(np.load(os.path.join(data_path, "pids_info_training.npy")))

input_xgrid = x_q2_inputs[:, 0]
input_q2grid = x_q2_inputs[:, 1]

base_path = "outputs/multimodel/"

# Descriptive names for the evaluated models
model_names = [
    "Model 1 (Base)",
    "Model 2 (Epochs)",
    "Model 3 (Wider)",
    "Model 4 (Deeper)",
    "Model 5 (One Dense)",
    "Model 6 (Linear)",
    "Model 7 (NNPDF)",
]

models = {}

# Load model predictions from .npz files
for i, name in enumerate(model_names, 1):
    file_path = os.path.join(base_path, f"model{i}.npz")
    temp_npz = np.load(file_path)
    models[name] = temp_npz["predictions"]

pid_cols = {pid: i for i, pid in enumerate(pids)}
# Define the flavor basis (PIDs)
output_basis = [-4, -3, -2, -1, 21, 1, 2, 3, 4]
noutput = len(output_basis)

output_data = np.zeros((len(val_pdf), noutput))

# Map PID columns to the output data array
for j, pid in enumerate(output_basis):
    col_idx = pid_cols[pid]
    output_data[:, j] = val_pdf[:, col_idx]

# 1. Mask selection and preparation
# Adjust Q2 value for specific energy scale evaluation (e.g., Z boson scale)
available_q2 = np.unique(input_q2grid)
q_target = 91.0  # GeV
q2_target = q_target**2
# Find the closest matching Q2 in the grid
q2_val = available_q2[np.abs(available_q2 - q2_target).argmin()]

# Apply mask for the selected Q2 scale
mask = np.isclose(input_q2grid, q2_val, atol=1e-5)

# Sort by Bjorken-x to ensure correct integration
x_subset = input_xgrid[mask]
sort_idx = np.argsort(x_subset)

# Define parton flavor mapping
idx_partons = {
    "g": 4,
    "d": 5,
    "u": 6,
    "s": 7,
    "c": 8,
    "d_bar": 3,
    "u_bar": 2,
    "s_bar": 1,
    "c_bar": 0,
}

results = {"Parton Flavor": list(idx_partons.keys())}

# 2. Iterate through models to calculate KL Divergence
for model_name, predictions in tqdm(models.items(), desc="Calculating KL Divergence"):
    kl_values = []

    for flavor, col in idx_partons.items():
        # Extract and sort values for the current flavor and scale
        y_true = output_data[mask][sort_idx, col]
        y_pred = predictions[mask][sort_idx, col]
        x_points = input_xgrid[mask][sort_idx]

        # Numerical stability: prevent log(0) or division by zero
        y_true = np.maximum(y_true, 1e-12)
        y_pred = np.maximum(y_pred, 1e-12)

        # Pointwise KL Divergence calculation: P * log(P/Q)
        kl_pointwise = y_true * np.log(y_true / y_pred)

        # Integrate over x-space using the trapezoidal rule
        kl = np.abs(np.trapz(kl_pointwise, x_points))
        kl_values.append(kl)

    results[model_name] = kl_values

# 3. Create DataFrame and export
df_results = pd.DataFrame(results)

# Set float format for console output
pd.options.display.float_format = "{:.5f}".format

print(f"\n Results at Q = {np.sqrt(q2_val):.2f} GeV:")
print(df_results.to_string(index=False))